In [1]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 1 — Install packages                                   ║
# ║  RUN THIS CELL ALONE FIRST — it will auto-restart runtime    ║
# ╚══════════════════════════════════════════════════════════════╝
import subprocess, sys, os

def pip(*pkgs):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *pkgs])

# No pinned numpy/pandas — let Colab use its pre-installed compatible versions.
# Pinning old numpy/pandas causes the 'mtrand ABI mismatch' ValueError.
pip(
    "datasets>=2.18.0",
    "transformers>=4.40.0",
    "sentence-transformers>=2.7.0",
    "scikit-learn>=1.4.0",
    "tqdm>=4.66.0",
    "accelerate>=0.26.0",
    "evaluate",
)

print("✅ Packages installed — restarting runtime now …")
os.kill(os.getpid(), 9)  # auto-restart; Colab reconnects in ~5 s

: 

: 

: 

In [5]:
!pip install -q "datasets==2.21.0"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.3/527.3 kB 16.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 177.6/177.6 kB 20.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2025.3.0 requires fsspec==2025.3.0, but you have fsspec 2024.6.1 which is incompatible.


In [1]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 2 — Mount Drive + GPU check                            ║
# ║  Run AFTER the runtime has restarted                         ║
# ╚══════════════════════════════════════════════════════════════╝
from google.colab import drive
drive.mount("/content/drive")

import torch
print("CUDA available:", torch.cuda.is_available())
print("Device name:   ", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")
assert torch.cuda.is_available(), "❌ No GPU — set Runtime type to T4 GPU!"
DEVICE = 0

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
CUDA available: True
Device name:    Tesla T4


In [2]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 3 — Config                                             ║
# ╚══════════════════════════════════════════════════════════════╝
import os

CNLI_SIZE  = 6820    # full ContractNLI train split
MNLI_SIZE  = 50000   # pool; genre filter keeps ~8-10k government rows
SYNTH_SIZE = 1000    # synthetic contradiction pairs

BASE_MODEL = "typeform/distilbert-base-uncased-mnli"
OUTPUT_DIR = "/content/drive/MyDrive/Athernex/nli_contract_model_final"
EPOCHS     = 5
BATCH_SIZE = 32      # T4 handles 32 at max_length=128
LR         = 2e-5
MAX_LEN    = 128

LABEL2ID = {"entailment": 0, "contradiction": 1, "neutral": 2}
ID2LABEL = {v: k for k, v in LABEL2ID.items()}

os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"✅ Config ready  |  output → {OUTPUT_DIR}")

✅ Config ready  |  output → /content/drive/MyDrive/Athernex/nli_contract_model_final


In [3]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 4 — Data loading helpers                               ║
# ╚══════════════════════════════════════════════════════════════╝
import re
import pandas as pd
from datasets import load_dataset

def clean_clause(text: str) -> str:
    text = re.sub(r'\s+', ' ', text).strip()
    text = re.sub(r'[^\x00-\x7F]+', '', text)
    return text

def load_contract_nli(split: str = "train", size: int = CNLI_SIZE):
    """Full ContractNLI — kiddothe2b/contract-nli, subset contractnli_a."""
    slice_str = f"{split}[:{size}]" if size else split
    return load_dataset(
        "kiddothe2b/contract-nli", "contractnli_a",
        split=slice_str, trust_remote_code=True
    )

def process_contract_nli(dataset) -> pd.DataFrame:
    """kiddothe2b schema: 0=contradiction, 1=entailment, 2=neutral."""
    label_map = {0: "contradiction", 1: "entailment", 2: "neutral"}
    records = []
    for s in dataset:
        p = clean_clause(s["premise"])
        h = clean_clause(s["hypothesis"])
        if len(p) < 20 or len(h) < 20:
            continue
        records.append({"clause1": p, "clause2": h,
                         "label": label_map.get(s["label"], "neutral")})
    return pd.DataFrame(records)

def load_mnli_government(split: str = "train", size: int = MNLI_SIZE):
    """MultiNLI filtered to government genre."""
    if split == "validation":
        split = "validation_matched"
    slice_str = f"{split}[:{size}]" if size else split
    ds = load_dataset("nyu-mll/multi_nli", split=slice_str, trust_remote_code=True)
    return ds.filter(lambda x: x["genre"] == "government")

def process_mnli_government(dataset) -> pd.DataFrame:
    """MultiNLI schema: 0=entailment, 1=neutral, 2=contradiction."""
    label_map = {0: "entailment", 1: "neutral", 2: "contradiction"}
    records = []
    for s in dataset:
        if not s["premise"] or not s["hypothesis"]:
            continue
        p = clean_clause(s["premise"])
        h = clean_clause(s["hypothesis"])
        if len(p) < 20 or len(h) < 20:
            continue
        records.append({"clause1": p, "clause2": h,
                         "label": label_map.get(s["label"], "neutral")})
    return pd.DataFrame(records)

NEGATION_MAP = {
    "shall": "shall not", "must": "must not",
    "will": "will not",   "may": "may not",
    "is required to": "is not required to",
    "exclusive": "non-exclusive", "limited": "unlimited",
    "terminate": "not terminate",
}

def simulate_contradiction(clause: str):
    for term, negated in NEGATION_MAP.items():
        if term in clause.lower():
            return re.sub(term, negated, clause, count=1, flags=re.IGNORECASE)
    return None

def build_synthetic_pairs(clauses: list, sample_size: int = SYNTH_SIZE) -> pd.DataFrame:
    import random; random.seed(42)
    sampled = random.sample(clauses, min(sample_size, len(clauses)))
    records = []
    for clause in sampled:
        neg = simulate_contradiction(clause)
        if neg:
            records.append({"clause1": clause, "clause2": neg, "label": "contradiction"})
    return pd.DataFrame(records)

print("✅ Data helpers defined.")

✅ Data helpers defined.


In [4]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 5 — Build full training dataset                        ║
# ╚══════════════════════════════════════════════════════════════╝
print("\n" + "="*55)
print("BUILDING FULL TRAINING DATA")
print("="*55)

print(f"\n[1/3] ContractNLI (size={CNLI_SIZE}) ...")
cnli_raw = load_contract_nli(size=CNLI_SIZE)
cnli_df  = process_contract_nli(cnli_raw)
print(f"  → {len(cnli_df)} pairs")
print(cnli_df["label"].value_counts().to_string())

print(f"\n[2/3] MultiNLI government (pool={MNLI_SIZE}) ...")
mnli_raw = load_mnli_government(size=MNLI_SIZE)
mnli_df  = process_mnli_government(mnli_raw)
print(f"  → {len(mnli_df)} pairs")
print(mnli_df["label"].value_counts().to_string())

print(f"\n[3/3] Synthetic contradictions (size={SYNTH_SIZE}) ...")
synth_df = build_synthetic_pairs(cnli_df["clause1"].tolist(), sample_size=SYNTH_SIZE)
print(f"  → {len(synth_df)} pairs")

valid_labels = {"entailment", "contradiction", "neutral"}
full_df = (
    pd.concat([cnli_df, mnli_df, synth_df], ignore_index=True)
    .sample(frac=1, random_state=42)
)
full_df = full_df[full_df["label"].isin(valid_labels)].dropna(
    subset=["clause1", "clause2", "label"])

print(f"\n{'='*55}")
print(f"✅ Total training pairs: {len(full_df)}")
print(full_df["label"].value_counts().to_string())
print("="*55)


BUILDING FULL TRAINING DATA

[1/3] ContractNLI (size=6820) ...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:103: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


Generating train split:   0%|          | 0/6819 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1991 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/978 [00:00<?, ? examples/s]

...

Generating train split:   0%|          | 0/392702 [00:00<?, ? examples/s]

Generating validation_matched split:   0%|          | 0/9815 [00:00<?, ? examples/s]

Generating validation_mismatched split:   0%|          | 0/9832 [00:00<?, ? examples/s]

Filter:   0%|          | 0/50000 [00:00<?, ? examples/s]

  → 9937 pairs
label
contradiction    3526
entailment       3318
neutral          3093

[3/3] Synthetic contradictions (size=1000) ...
  → 842 pairs

✅ Total training pairs: 17598
label
entailment       6513
neutral          5913
contradiction    5172


In [5]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 6 — Tokenize & split                                   ║
# ╚══════════════════════════════════════════════════════════════╝
from datasets import Dataset
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)

df_train = full_df.copy()
df_train["label"] = df_train["label"].map(LABEL2ID)

hf_ds  = Dataset.from_pandas(df_train[["clause1", "clause2", "label"]])
splits = hf_ds.train_test_split(test_size=0.15, seed=42)

def tokenize_fn(batch):
    return tokenizer(
        batch["clause1"], batch["clause2"],
        truncation=True, padding="max_length", max_length=MAX_LEN
    )

tokenized = splits.map(tokenize_fn, batched=True, batch_size=256)
# Drop raw text columns — keep only model inputs + label
tokenized = tokenized.remove_columns(["clause1", "clause2"])
tokenized.set_format("torch")

print(f"Train: {len(tokenized['train'])}  |  Eval: {len(tokenized['test'])}")

config.json:   0%|          | 0.00/776 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/258 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

Map:   0%|          | 0/14958 [00:00<?, ? examples/s]

Map:   0%|          | 0/2640 [00:00<?, ? examples/s]

Train: 14958  |  Eval: 2640


In [6]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 7 — Model & TrainingArguments                          ║
# ╚══════════════════════════════════════════════════════════════╝
from transformers import (
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
)
import numpy as np
from sklearn.metrics import f1_score, accuracy_score

model = AutoModelForSequenceClassification.from_pretrained(
    BASE_MODEL,
    num_labels=3,
    id2label=ID2LABEL,
    label2id=LABEL2ID,
    ignore_mismatched_sizes=True,
)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1": f1_score(labels, preds, average="weighted", zero_division=0),
    }

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    learning_rate=LR,
    weight_decay=0.01,
    warmup_ratio=0.1,
    lr_scheduler_type="cosine",
    eval_strategy="epoch",       # ← replaces deprecated evaluation_strategy
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    logging_steps=50,
    fp16=True,                   # T4 supports FP16 — ~2x speed boost
    dataloader_num_workers=2,
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["test"],
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

print("✅ Model & Trainer ready.")

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


✅ Model & Trainer ready.


In [7]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 8 — Train (~20-35 min on T4)                           ║
# ╚══════════════════════════════════════════════════════════════╝
print("\n🚀 Starting training ...")
train_result = trainer.train()
print("\n✅ Training complete!")
print(train_result.metrics)


🚀 Starting training ...


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.331108,0.299753,0.882197,0.883166
2,0.287276,0.249932,0.909470,0.909536
3,0.193082,0.232398,0.920076,0.919938
4,0.181951,0.229638,0.923485,0.923234
5,0.148084,0.244799,0.921591,0.921433


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].



✅ Training complete!
{'train_runtime': 267.0102, 'train_samples_per_second': 280.102, 'train_steps_per_second': 8.764, 'total_flos': 2476853356746240.0, 'train_loss': 0.3612339029964219, 'epoch': 5.0}


In [8]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 9 — Save model to Drive                                ║
# ╚══════════════════════════════════════════════════════════════╝
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"\n✅ Model saved → {OUTPUT_DIR}")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


✅ Model saved → /content/drive/MyDrive/Athernex/nli_contract_model_final


In [9]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 10 — Final eval on validation split                    ║
# ╚══════════════════════════════════════════════════════════════╝
print("\n📊 Final evaluation on validation split ...")
eval_results = trainer.evaluate()
print(eval_results)


📊 Final evaluation on validation split ...


{'eval_loss': 0.22964176535606384, 'eval_accuracy': 0.9234848484848485, 'eval_f1': 0.9232344709378449, 'eval_runtime': 2.2543, 'eval_samples_per_second': 1171.072, 'eval_steps_per_second': 36.818, 'epoch': 5.0}


In [10]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 11 — Held-out test (MultiNLI validation_matched)       ║
# ╚══════════════════════════════════════════════════════════════╝
from sklearn.metrics import classification_report, confusion_matrix
from transformers import pipeline as hf_pipeline

print("\n[Held-out] Loading MultiNLI validation_matched (government) ...")
val_raw = load_mnli_government(split="validation_matched", size=5000)
val_df  = process_mnli_government(val_raw)
val_df  = val_df[val_df["label"].isin(valid_labels)].dropna().reset_index(drop=True)
print(f"  → {len(val_df)} held-out samples")

# Batch pipeline — no top_k so batch_size works cleanly
batch_pipe = hf_pipeline(
    "text-classification",
    model=OUTPUT_DIR,
    tokenizer=tokenizer,
    device=0,
    batch_size=64,
    truncation=True,
    max_length=MAX_LEN,
)

texts       = [f"{r.clause1} [SEP] {r.clause2}" for r in val_df.itertuples()]
raw_preds   = batch_pipe(texts)
pred_labels = [p["label"] for p in raw_preds]  # top-1 label per sample

print("\n" + "="*60)
print("HELD-OUT CLASSIFICATION REPORT")
print("="*60)
print(classification_report(val_df["label"].tolist(), pred_labels))

cm = confusion_matrix(
    val_df["label"].tolist(), pred_labels,
    labels=["entailment", "contradiction", "neutral"]
)
print("Confusion Matrix:")
print(pd.DataFrame(
    cm,
    index=["True: entailment", "True: contradiction", "True: neutral"],
    columns=["Pred: entailment", "Pred: contradiction", "Pred: neutral"],
))


[Held-out] Loading MultiNLI validation_matched (government) ...


Filter:   0%|          | 0/5000 [00:00<?, ? examples/s]

  → 996 held-out samples


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]


HELD-OUT CLASSIFICATION REPORT
               precision    recall  f1-score   support

contradiction       0.85      0.86      0.86       335
   entailment       0.90      0.84      0.87       354
      neutral       0.78      0.83      0.80       307

     accuracy                           0.84       996
    macro avg       0.84      0.84      0.84       996
 weighted avg       0.85      0.84      0.85       996

Confusion Matrix:
                     Pred: entailment  Pred: contradiction  Pred: neutral
True: entailment                  299                   17             38
True: contradiction                12                  288             35
True: neutral                      21                   32            254


In [13]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 12 — Contract clause smoke test (challenging)          ║
# ╚══════════════════════════════════════════════════════════════╝
from transformers import pipeline as hf_pipeline

EXAMPLE_PAIRS = [
    # --- CONTRADICTIONS: subtle conflicts ---
    ("The agreement shall automatically renew for successive one-year terms unless terminated with 90 days prior written notice.",
     "This agreement expires on the end date and shall not renew automatically under any circumstances.",
     "contradiction"),

    ("Seller warrants that all deliverables shall be free from defects for a period of 24 months from acceptance.",
     "Seller disclaims all warranties, express or implied, including any warranty of merchantability or fitness.",
     "contradiction"),

    ("The Licensee is granted an exclusive, worldwide, perpetual license to use the Software.",
     "The license granted herein is non-exclusive, limited to the United States, and valid for 12 months only.",
     "contradiction"),

    ("All disputes arising under this agreement shall be resolved through binding arbitration in New York.",
     "Either party may bring suit in any court of competent jurisdiction to resolve disputes under this agreement.",
     "contradiction"),

    ("The contractor shall maintain professional liability insurance with coverage of no less than $5,000,000.",
     "The contractor is not required to carry any form of professional liability insurance.",
     "contradiction"),

    # --- ENTAILMENTS: same meaning, different wording ---
    ("Neither party shall disclose Confidential Information to any third party without prior written consent.",
     "Confidential Information must not be shared with outside parties unless the disclosing party agrees in writing.",
     "entailment"),

    ("The Company shall indemnify and hold harmless the Consultant against all claims arising from the Company's negligence.",
     "If claims arise due to the Company's negligent acts, the Company is responsible for indemnifying the Consultant.",
     "entailment"),

    ("Force majeure events including natural disasters, war, and pandemic shall excuse performance obligations.",
     "Obligations under this contract are suspended during events such as pandemics, wars, or natural disasters beyond the parties' control.",
     "entailment"),

    # --- NEUTRALS: related but independent clauses ---
    ("The governing law of this agreement shall be the laws of the State of Delaware.",
     "All invoices must be submitted within 30 days of service completion.",
     "neutral"),

    ("Employee agrees to a 12-month non-compete restriction within a 50-mile radius.",
     "The company reserves the right to modify employee benefits at its sole discretion.",
     "neutral"),

    ("The data processor shall implement AES-256 encryption for all data at rest.",
     "The data controller shall conduct annual privacy impact assessments.",
     "neutral"),

    # --- HARD CASES: tricky edge cases ---
    ("Payment of the full contract price is due upon delivery of the final deliverable.",
     "Payment shall be made in four equal quarterly installments over the term of the agreement.",
     "contradiction"),

    ("Either party may terminate this agreement for convenience upon 30 days written notice.",
     "This agreement may only be terminated for cause, specifically material breach that remains uncured for 60 days.",
     "contradiction"),

    ("The supplier shall deliver all goods FOB destination, with risk of loss transferring upon delivery.",
     "All shipments are FOB origin; risk of loss passes to the buyer when goods are tendered to the carrier.",
     "contradiction"),

    ("Consultant retains all intellectual property rights in pre-existing materials incorporated into the deliverables.",
     "All work product, including any pre-existing materials used therein, shall be considered work-for-hire owned exclusively by the Client.",
     "contradiction"),
]

smoke_pipe = hf_pipeline(
    "text-classification",
    model=OUTPUT_DIR,
    tokenizer=tokenizer,
    device=0,
    top_k=None,
    truncation=True,
    max_length=MAX_LEN,
)

print("\n" + "="*60)
print("CONTRACT CLAUSE SMOKE TEST (CHALLENGING)")
print("="*60)
correct = 0
for clause1, clause2, expected in EXAMPLE_PAIRS:
    result = smoke_pipe(f"{clause1} [SEP] {clause2}")
    if result and isinstance(result[0], list):
        result = result[0]
    scores = {r["label"]: r["score"] for r in result}
    got    = max(scores, key=scores.get)
    mark   = "✓" if got == expected else "✗"
    correct += (got == expected)
    print(f"\n{mark}  Expected: {expected:15s} | Got: {got:15s} ({scores[got]:.2%})")
    print(f"   C1: {clause1[:80]}")
    print(f"   C2: {clause2[:80]}")

print(f"\n{'='*60}")
print(f"Smoke test accuracy: {correct}/{len(EXAMPLE_PAIRS)}")
print("="*60)
print("\n🎉 All done! Model saved to Google Drive.")


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset



CONTRACT CLAUSE SMOKE TEST (CHALLENGING)

✓  Expected: contradiction   | Got: contradiction   (99.81%)
   C1: The agreement shall automatically renew for successive one-year terms unless ter
   C2: This agreement expires on the end date and shall not renew automatically under a

✗  Expected: contradiction   | Got: neutral         (41.68%)
   C1: Seller warrants that all deliverables shall be free from defects for a period of
   C2: Seller disclaims all warranties, express or implied, including any warranty of m

✓  Expected: contradiction   | Got: contradiction   (99.94%)
   C1: The Licensee is granted an exclusive, worldwide, perpetual license to use the So
   C2: The license granted herein is non-exclusive, limited to the United States, and v

✗  Expected: contradiction   | Got: neutral         (99.44%)
   C1: All disputes arising under this agreement shall be resolved through binding arbi
   C2: Either party may bring suit in any court of competent jurisdiction to resolve di

✓  Ex